# Benchmark: .pt (PyTorch) vs ONNX throughput & latency

This notebook compares inference **latency** (per image) and **throughput** (images/sec) between the PyTorch `.pt`/`.pth` checkpoints and the exported ONNX models for the Devanagari OCR pipeline.

## Model sources (local paths only)

The `.pt` checkpoints and ONNX files are **not** downloaded by this notebook - you must place them locally. Update the paths in the config cell below.

- `.pt` recognition checkpoints: `iter_60000.pth` (CTC) and `iter_70000.pth` (Attn)
- ONNX recognition: `ResNetBiLSTMCTCv1.onnx`, `ResNetBiLSTMAttnv1.onnx`
- ONNX detection: `LineDetectionv4.onnx`

> Note: the released ONNX recognition models are exported with a **fixed batch size of 1** (the `dynamic_axes` were commented out during export). That is why recognition ONNX batch throughput below is sequential. See the *Dynamic-batch ONNX export* section at the end.

In [ ]:
import os
import sys
import time
from collections import OrderedDict

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

import onnxruntime as ort
import matplotlib.pyplot as plt

sys.path.append('./')
from modules.model import Model
from modules.utils import CTCLabelConverter, AttnLabelConverter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# CONFIG: point these at your local files. Nothing is downloaded.
# ============================================================
class Paths:
    # .pt recognition checkpoints
    pt_ctc = 'models/iter_60000.pth'          # CTC  checkpoint
    pt_attn = 'models/iter_70000.pth'         # Attn checkpoint

    # ONNX recognition models
    onnx_ctc = 'models/ResNetBiLSTMCTCv1.onnx'
    onnx_attn = 'models/ResNetBiLSTMAttnv1.onnx'

    # ONNX detection model
    onnx_detection = 'models/LineDetectionv4.onnx'

    # Directory containing cropped text-line images for recognition benchmarks
    crops_dir = 'crops'

    # Sample full-page image for the detection benchmark (optional)
    page_image = 'images/202_page.jpg'

P = Paths()

# Benchmark knobs
NUM_WARMUP = 5
NUM_ITERS = 50          # latency benchmark iterations
BATCH_SIZES = [1, 4, 8, 16]   # throughput benchmark (only meaningful for dynamic-batch models)

crop_files = [
    os.path.join(P.crops_dir, f)
    for f in sorted(os.listdir(P.crops_dir))
    if os.path.isfile(os.path.join(P.crops_dir, f))
] if os.path.isdir(P.crops_dir) else []
print(f'Found {len(crop_files)} crop images in {P.crops_dir}')

In [ ]:
# ============================================================
# Config object + model loading (reused from inference.ipynb)
# ============================================================
class NepaliTextRecognitionConfig:
    def __init__(self):
        self.number = '०१२३४५६७८९0123456789'
        self.symbol = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{}~।॥—‘’“”… "
        self.lang_char = 'अआइईउऊऋएऐओऔअंअःकखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहक्षत्रज्ञािीुूृेैोौंःँॅॉ'

        self.character = self.number + self.symbol + self.lang_char

        self.Transformation = 'None'
        self.FeatureExtraction = 'ResNet'
        self.SequenceModeling = 'BiLSTM'
        self.Prediction = 'Attn'

        self.imgH = 80
        self.imgW = 1220
        self.input_channel = 3
        self.output_channel = 256
        self.hidden_size = 256
        self.num_fiducial = 20

        self.batch_max_length = 200
        self.sensitive = True
        self.PAD = False
        self.rgb = True
        self.contrast_adjust = False
        self.decode = 'greedy'
        self.num_class = len(self.character)

config = NepaliTextRecognitionConfig()
print(f'Character set size: {config.num_class}')
print(f'Input: {config.input_channel}x{config.imgH}x{config.imgW}')

In [ ]:
def load_crnn_model(model_path, config, prediction):
    """Load a PyTorch CRNN model + its label converter."""
    config.Prediction = prediction
    if prediction == 'CTC':
        converter = CTCLabelConverter(config.character)
    elif prediction == 'Attn':
        converter = AttnLabelConverter(config.character)
    else:
        raise ValueError(prediction)
    config.num_class = len(converter.character)

    model = Model(config)
    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)
    state_dict = torch.load(model_path, map_location=device)
    if isinstance(state_dict, dict) and 'state_dict' in state_dict:
        state_dict = state_dict['state_dict']
    if all(k.startswith('module.') for k in state_dict.keys()):
        state_dict = OrderedDict((k[7:], v) for k, v in state_dict.items())
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model, converter


def center_and_resize_image(img, target_size=(1220, 80)):
    if isinstance(img, str):
        img = Image.open(img)
    target_w, target_h = target_size
    if img.width > target_w or img.height > target_h:
        img.thumbnail((target_w, target_h), Image.LANCZOS)
    new_img = Image.new("RGB", (target_w, target_h), color="black")
    paste_x = (target_w - img.width) // 2
    paste_y = (target_h - img.height) // 2
    new_img.paste(img, (paste_x, paste_y))
    return new_img


def preprocess_crnn_image(image_path, config, return_tensor=True):
    """Preprocess a single crop to (1, C, H, W) tensor."""
    processed_pil = center_and_resize_image(image_path, (config.imgW, config.imgH))
    if not return_tensor:
        return processed_pil
    if config.input_channel == 1:
        image_np = np.array(processed_pil.convert('L')).astype(np.float32) / 255.0
        image_tensor = torch.from_numpy(image_np).unsqueeze(0)
        image_tensor = (image_tensor - 0.5) / 0.5
    else:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        image_tensor = transform(processed_pil)
    return image_tensor.unsqueeze(0)


def preprocess_crop_batch(paths, config):
    """Stack preprocessed crops into a (B, C, H, W) tensor."""
    tensors = [preprocess_crnn_image(p, config)[0] for p in paths]
    return torch.stack(tensors, dim=0)

In [ ]:
# ============================================================
# ONNX session + .pt forward helpers
# ============================================================
def load_onnx(path, providers=None):
    if providers is None:
        providers = ort.get_available_providers()
    return ort.InferenceSession(path, providers=providers)


def pt_forward_ctc(model, batch_tensor):
    """Run a .pt CTC model on a (B,C,H,W) tensor, return logits."""
    batch_size = batch_tensor.size(0)
    text = torch.LongTensor(batch_size, config.batch_max_length).fill_(0).to(device)
    with torch.no_grad():
        return model(batch_tensor.to(device), text, is_train=False)


def pt_forward_attn(model, batch_tensor):
    """Run a .pt Attn model on a (B,C,H,W) tensor, return logits."""
    batch_size = batch_tensor.size(0)
    text = torch.LongTensor(batch_size, config.batch_max_length + 1).fill_(0).to(device)
    with torch.no_grad():
        return model(batch_tensor.to(device), text, is_train=False)


def onnx_forward(session, batch_np):
    """Run an ONNX recognition session on a (B,C,H,W) float32 array."""
    return session.run(None, {"input": batch_np.astype(np.float32)})[0]

In [ ]:
# ============================================================
# Load all models
# ============================================================
loaded = {}

# .pt models
loaded['pt_ctc'] = load_crnn_model(P.pt_ctc, config, 'CTC')
loaded['pt_attn'] = load_crnn_model(P.pt_attn, config, 'Attn')

# ONNX models
loaded['onnx_ctc'] = load_onnx(P.onnx_ctc)
loaded['onnx_attn'] = load_onnx(P.onnx_attn)

for k, v in loaded.items():
    print(f'{k}: loaded')

# Show the ONNX input shapes (confirms whether batch is dynamic or fixed)
for name in ['onnx_ctc', 'onnx_attn']:
    for inp in loaded[name].get_inputs():
        print(f'{name} input "{inp.name}" shape={inp.shape}')

In [ ]:
# ============================================================
# Latency benchmark (per image, batch=1)
# ============================================================
def bench_latency(fn, n_warmup=NUM_WARMUP, n_iter=NUM_ITERS):
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_iter):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    return {
        'mean_ms': float(times.mean() * 1e3),
        'median_ms': float(np.median(times) * 1e3),
        'p95_ms': float(np.percentile(times, 95) * 1e3),
        'images_per_sec': 1.0 / float(times.mean()),
    }

# Single fixed crop
crop = crop_files[0]
print(f'Benchmarking on crop: {crop}')
img_t = preprocess_crnn_image(crop, config).to(device)
img_np = img_t.cpu().numpy()

latency_results = {}

latency_results['pt_ctc'] = bench_latency(
    lambda: pt_forward_ctc(loaded['pt_ctc'][0], img_t))
latency_results['onnx_ctc'] = bench_latency(
    lambda: onnx_forward(loaded['onnx_ctc'], img_np))
latency_results['pt_attn'] = bench_latency(
    lambda: pt_forward_attn(loaded['pt_attn'][0], img_t))
latency_results['onnx_attn'] = bench_latency(
    lambda: onnx_forward(loaded['onnx_attn'], img_np))

import pandas as pd
lat_df = pd.DataFrame(latency_results).T
lat_df.index.name = 'model'
print('\nLatency (per image, batch=1):')
display(lat_df.round(3))

In [ ]:
# ============================================================
# Throughput benchmark (images/sec) at various batch sizes
# ============================================================
def bench_throughput(fn_factory, n_iter=NUM_ITERS):
    """fn_factory(batch_size) returns a callable running one batch."""
    results = {}
    for bs in BATCH_SIZES:
        # pad the crop list to a full batch
        paths = [crop_files[i % len(crop_files)] for i in range(bs)]
        fn = fn_factory(paths)
        for _ in range(NUM_WARMUP):
            fn()
        times = []
        for _ in range(n_iter):
            t0 = time.perf_counter()
            fn()
            times.append(time.perf_counter() - t0)
        times = np.array(times)
        results[bs] = float(bs / times.mean())
    return results

# .pt models: true batching
pt_ctc_model = loaded['pt_ctc'][0]
pt_attn_model = loaded['pt_attn'][0]

thru = {}
thru['pt_ctc'] = bench_throughput(
    lambda paths: (lambda b: pt_forward_ctc(pt_ctc_model, b))(preprocess_crop_batch(paths, config)))
thru['pt_attn'] = bench_throughput(
    lambda paths: (lambda b: pt_forward_attn(pt_attn_model, b))(preprocess_crop_batch(paths, config)))

# ONNX recognition is fixed batch=1 -> sequential (loop over batch)
def onnx_seq(session):
    def make(paths):
        batch = preprocess_crop_batch(paths, config).cpu().numpy()
        def run():
            for i in range(batch.shape[0]):
                onnx_forward(session, batch[i:i+1])
        return run
    return make

thru['onnx_ctc'] = bench_throughput(onnx_seq(loaded['onnx_ctc']))
thru['onnx_attn'] = bench_throughput(onnx_seq(loaded['onnx_attn']))

thru_df = pd.DataFrame(thru).T
thru_df.index.name = 'model'
thru_df.columns = [f'bs={bs}' for bs in BATCH_SIZES]
print('\nThroughput (images/sec):')
display(thru_df.round(2))

In [ ]:
# ============================================================
# Plots: latency and throughput comparison
# ============================================================
labels = ['pt_ctc', 'onnx_ctc', 'pt_attn', 'onnx_attn']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Latency bar chart
axes[0].bar(labels, [latency_results[l]['mean_ms'] for l in labels],
            color=['#4c72b0', '#dd8452', '#4c72b0', '#dd8452'])
axes[0].set_title('Mean latency per image (ms)')
axes[0].set_ylabel('ms')
axes[0].tick_params(axis='x', rotation=45)
for i, l in enumerate(labels):
    axes[0].text(i, latency_results[l]['mean_ms'], f"{latency_results[l]['mean_ms']:.1f}",
                 ha='center', va='bottom')

# Throughput line chart (images/sec)
x = BATCH_SIZES
for name in ['pt_ctc', 'onnx_ctc', 'pt_attn', 'onnx_attn']:
    axes[1].plot(x, [thru[name][bs] for bs in x], marker='o', label=name)
axes[1].set_title('Throughput vs batch size (images/sec)')
axes[1].set_xlabel('batch size')
axes[1].set_ylabel('images/sec')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary table
# ============================================================
summary = pd.DataFrame({
    'model': labels,
    'latency_mean_ms': [latency_results[l]['mean_ms'] for l in labels],
    'latency_p95_ms': [latency_results[l]['p95_ms'] for l in labels],
    'throughput_bs1_img_s': [thru[l][1] for l in labels],
})
print('Summary:')
display(summary.round(3))

print('\nInterpretation:')
print('- ONNX is generally faster per image than the .pt model on the same hardware.')
print('- The .pt models support true batching (throughput rises with batch size).')
print('- The released ONNX recognition models are fixed batch=1, so ONNX throughput is')
print('  sequential across the batch (flat curve). Export a dynamic-batch ONNX to enable batching.')
print('  See the Dynamic-batch ONNX export section below.')

In [ ]:
# ============================================================
# Optional: detection model benchmark (ONNX)
# ============================================================
def letterbox(img, new_shape=(1024, 1024), color=(114, 114, 114)):
    orig_w, orig_h = img.size
    r = min(new_shape[0] / orig_h, new_shape[1] / orig_w)
    new_unpad = int(orig_w * r), int(orig_h * r)
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2
    img_resized = img.resize(new_unpad, Image.BILINEAR)
    new_img = Image.new("RGB", new_shape, color)
    new_img.paste(img_resized, (int(dw), int(dh)))
    return new_img, new_unpad[0], new_unpad[1], int(dw), int(dh), r

if os.path.isfile(P.onnx_detection) and os.path.isfile(P.page_image):
    det_sess = load_onnx(P.onnx_detection)
    in_name = det_sess.get_inputs()[0].name

    img = Image.open(P.page_image).convert('RGB')
    padded, *_ = letterbox(img, (1024, 1024))
    arr = np.array(padded).astype(np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[None, ...]

    for _ in range(NUM_WARMUP):
        det_sess.run(None, {in_name: arr})
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        det_sess.run(None, {in_name: arr})
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    print(f'Detection ONNX ({os.path.basename(P.onnx_detection)}): '
          f'mean {times.mean()*1e3:.1f} ms/image, '
          f'p95 {np.percentile(times,95)*1e3:.1f} ms')
else:
    print('Detection benchmark skipped: set P.onnx_detection and P.page_image to valid files.')

In [ ]:
# ============================================================
# Dynamic-batch ONNX export (fix)
# ============================================================
# The released ONNX recognition models are fixed batch=1 because the
# dynamic_axes were commented out during export. Here is the corrected
# export that produces a model accepting any batch size.

import torch.onnx
import onnx
from onnx import checker

def export_crnn_to_onnx_dynamic(model, converter, config, output_path,
                               batch_size=1, opset_version=11):
    """Export a CRNN model to ONNX with a DYNAMIC batch dimension."""
    model.eval()
    dummy_input = torch.randn(batch_size, config.input_channel,
                              config.imgH, config.imgW).to(device)

    if config.Prediction == 'CTC':
        # CTC only needs the image; 'text' is unused in the traced forward.
        dynamic_axes = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
        model_args = (dummy_input,)
        input_names = ['input']
    else:
        # Attn needs the 'text' input; its batch dim must also be dynamic.
        dummy_text = torch.LongTensor(batch_size, config.batch_max_length + 1).fill_(0).to(device)
        dynamic_axes = {
            'input': {0: 'batch_size'},
            'text': {0: 'batch_size'},
            'output': {0: 'batch_size'},
        }
        model_args = (dummy_input, dummy_text)
        input_names = ['input', 'text']

    with torch.no_grad():
        torch.onnx.export(
            model,
            model_args,
            output_path,
            export_params=True,
            opset_version=opset_version,
            do_constant_folding=True,
            input_names=input_names,
            output_names=['output'],
            dynamic_axes=dynamic_axes,
            verbose=False,
        )

    onnx_model = onnx.load(output_path)
    checker.check_model(onnx_model)
    print(f'Exported dynamic-batch model: {output_path}')
    return output_path

# Example usage:
# export_crnn_to_onnx_dynamic(loaded['pt_ctc'][0], loaded['pt_ctc'][1], config,
#                             'models/ResNetBiLSTMCTCv1Dynamic.onnx')
# export_crnn_to_onnx_dynamic(loaded['pt_attn'][0], loaded['pt_attn'][1], config,
#                             'models/ResNetBiLSTMAttnv1Dynamic.onnx')

In [ ]:
# ============================================================
# Verify a dynamic-batch ONNX model accepts multiple batch sizes
# ============================================================
def verify_dynamic_batch(onnx_path):
    sess = load_onnx(onnx_path)
    in_name = sess.get_inputs()[0].name
    print(f'Input shape: {sess.get_inputs()[0].shape}')
    ok = True
    for bs in [1, 4, 8]:
        try:
            batch = np.zeros((bs, config.input_channel, config.imgH, config.imgW),
                             dtype=np.float32)
            out = sess.run(None, {in_name: batch})[0]
            print(f'  batch={bs} -> output {out.shape} OK')
        except Exception as e:
            ok = False
            print(f'  batch={bs} FAILED: {e}')
    return ok

# Run on a freshly exported dynamic model, e.g.:
# verify_dynamic_batch('models/ResNetBiLSTMCTCv1Dynamic.onnx')